In [ ]:
import os
import random
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

# Target Unity Catalog Volume destination path
volume_dir = "/Volumes/maven_market_uc/gold/external_benchmarks"
volume_csv_path = f"{volume_dir}/external_market_benchmarks.csv"

# 1. Fetch all distinct product_ids from gold.dim_products to maintain referential integrity
try:
    product_rows = spark.table("maven_market_uc.gold.dim_products") \
        .select("product_id") \
        .filter("product_id IS NOT NULL") \
        .distinct() \
        .orderBy("product_id") \
        .collect()
    product_ids = [int(r.product_id) for r in product_rows]
    print(f"Loaded {len(product_ids)} distinct product IDs from gold.dim_products.")
except Exception as e:
    print(f"Falling back to default range (1 to 1560): {e}")
    product_ids = list(range(1, 1561))

# Deterministic seed for reproducible data across environments
random.seed(42)

# 2. Synthesize market benchmark metrics for all 1,560 products
benchmark_records = []
for pid in product_ids:
    benchmark_rev = round(random.uniform(350.0, 5200.0), 2)
    growth_pct = round(random.uniform(1.5, 9.8), 1)
    comp_idx = round(random.uniform(0.75, 1.35), 2)
    benchmark_records.append((pid, benchmark_rev, growth_pct, comp_idx))

# 3. Define schema and assemble DataFrame
schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("industry_benchmark_revenue", DoubleType(), False),
    StructField("market_expected_growth_pct", DoubleType(), False),
    StructField("competitor_index", DoubleType(), False)
])

df_benchmarks = spark.createDataFrame(benchmark_records, schema=schema)

# 4. Write as a single header-enabled CSV file directly to the Volume mount
os.makedirs(volume_dir, exist_ok=True)
df_pd = df_benchmarks.toPandas()
df_pd.to_csv(volume_csv_path, index=False)

print(f"Successfully generated and wrote {len(df_pd)} records to: {volume_csv_path}")

# 5. Immediate verification read
df_check = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(volume_csv_path)

display(df_check.summary())
print(f"Verified row count in volume: {df_check.count()}")